In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Techniky zmírnění a potlačení chyb

> **Note:** Beta verze nového modelu spouštění je nyní k dispozici. Řízený model spouštění poskytuje větší flexibilitu při přizpůsobování pracovního postupu zmírnění chyb. Více informací najdeš v průvodci [Řízený model spouštění](/guides/directed-execution-model).


<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme použít tyto verze nebo novější.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Techniky zmírnění chyb a potlačení chyb se používají ke zlepšení kvality výsledků při škálování na větší pracovní zátěže. Tato stránka poskytuje vysvětlení na vysoké úrovni technik potlačení a zmírnění chyb dostupných prostřednictvím Qiskit Runtime.

Následující buňka importuje primitiv Estimator a vytvoří Backend, který bude použit pro inicializaci Estimatoru v pozdějších buňkách kódu.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Dynamické oddělení
Kvantové obvody jsou na hardwaru IBM&reg; spouštěny jako sekvence mikrovlnných pulzů, které musí být naplánovány a spuštěny v přesných časových intervalech.
Bohužel nežádoucí interakce mezi Qubity mohou vést ke koherentním chybám na nečinných Qubitech. Dynamické oddělení funguje tak, že na nečinné Qubity vkládá sekvence pulzů, které přibližně ruší efekt těchto chyb. Každá vložená sekvence pulzů odpovídá identické operaci, ale fyzická přítomnost pulzů má účinek potlačení chyb.
Existuje mnoho možných voleb sekvencí pulzů a to, která sekvence je pro každý konkrétní případ lepší, zůstává [aktivní oblastí výzkumu](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Vezmi na vědomí, že dynamické oddělení je užitečné zejména pro Circuit obsahující mezery, ve kterých některé Qubity nečinně čekají bez jakýchkoliv operací, které by na ně působily. Pokud jsou operace v Circuit velmi hustě uspořádány tak, že všechny Qubity jsou většinu času zaneprázdněny, přidání pulzů dynamického oddělení nemusí zlepšit výkon. Ve skutečnosti by mohlo výkon i zhoršit kvůli nedokonalostem v samotných pulzech.

Diagram níže zobrazuje dynamické oddělení se sekvencí pulzů XX. Abstraktní Circuit vlevo je namapován na plán mikrovlnných pulzů vpravo nahoře. Vpravo dole je zobrazený stejný plán, ale s vloženou sekvencí dvou pulzů X v době nečinnosti prvního Qubitu.

![Zobrazení dynamického oddělení](../docs/images/guides/error-mitigation-explanation/dd.avif)

Dynamické oddělení lze zapnout nastavením `enable` na `True` v [možnostech dynamického oddělení](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). Možnost `sequence_type` lze použít k výběru z několika různých sekvencí pulzů. Výchozí typ sekvence je `"XX"`.

Následující buňka kódu ukazuje, jak zapnout dynamické oddělení pro Estimator a zvolit sekvenci dynamického oddělení.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Pauliho twirling
Twirling, známý také jako [randomizovaná kompilace](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), je široce používaná technika pro převod libovolných šumových kanálů na šumové kanály s konkrétnější strukturou.

Pauliho twirling je speciální druh twirling, který používá Pauliho operace. Má efekt transformace libovolného kvantového kanálu na Pauliho kanál. Samotný tento postup může zmírnit koherentní šum, protože koherentní šum má tendenci se akumulovat kvadraticky s počtem operací, zatímco Pauliho šum se akumuluje lineárně. Pauliho twirling se často kombinuje s jinými technikami zmírnění chyb, které fungují lépe s Pauliho šumem než s libovolným šumem.

Pauliho twirling se implementuje tak, že se zvolená sada Gate obklopí náhodně vybranými jednoqubitovými Pauliho Gate takovým způsobem, aby ideální efekt Gate zůstal stejný. Výsledkem je, že jeden Circuit je nahrazen náhodným souborem Circuit, všechny se stejným ideálním efektem. Při vzorkování Circuit jsou vzorky čerpány z více náhodných instancí, nikoli jen z jedné.

![Zobrazení Pauliho twirling](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Protože většina chyb v současném kvantovém hardwaru pochází z dvouqubitových Gate, tato technika se často aplikuje výhradně na (nativní) dvouqubitové Gate. Následující diagram zobrazuje některé Pauliho twirly pro Gate CNOT a ECR. Každý Circuit v řadě má stejný ideální efekt.

![Zobrazení Gate twirls](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Pauliho twirling lze zapnout nastavením `enable_gates` na `True` v [možnostech twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Další pozoruhodné možnosti zahrnují:

- `num_randomizations`: Počet instancí Circuit k vyčerpání ze souboru twirled Circuit.
- `shots_per_randomization`: Počet vzorků k odebrání z každé instance Circuit.

Následující buňka kódu ukazuje, jak zapnout Pauliho twirling a nastavit tyto možnosti pro estimator. Žádná z těchto možností nemusí být nastavena explicitně.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Twirled readout error extinction (TREX)
[Twirled readout error extinction (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) zmírňuje efekt chyb měření při odhadování střední hodnoty Pauliho observabil.
Vychází z konceptu twirled měření, která se provádí náhodnou náhradou měřicích Gate sekvencí (1) Pauliho X Gate, (2) měření a (3) překlopení klasického bitu. Stejně jako při standardním gate twirling je tato sekvence v nepřítomnosti šumu ekvivalentní prostému měření, jak je znázorněno v následujícím diagramu:

![Zobrazení measurement twirling](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

V přítomnosti chyby čtení má measurement twirling efekt diagonalizace matice přenosu chyby čtení, což usnadňuje její inverzi. Odhad matice přenosu chyby čtení vyžaduje provádění dodatečných kalibračních Circuit, což přináší malou režii.

TREX lze zapnout nastavením `measure_mitigation` na `True` v [možnostech odolnosti Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pro Estimator. Možnosti pro učení šumu měření jsou popsány [zde](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Stejně jako u gate twirling můžeš nastavit počet randomizací Circuit a počet vzorků na randomizaci.

Následující buňka kódu ukazuje, jak zapnout TREX a nastavit tyto možnosti pro estimator. Žádná z těchto možností nemusí být nastavena explicitně.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Extrapolace na nulový šum (ZNE)
Extrapolace na nulový šum (ZNE) je technika pro zmírnění chyb při odhadování střední hodnoty observabil. Přestože výsledky obvykle zlepšuje, není zaručeno, že vyprodukuje nestranný výsledek.

ZNE se skládá ze dvou fází:

1. _Zesilování šumu_: Původní kvantový Circuit se spouští vícekrát při různých úrovních šumu.
2. _Extrapolace_: Ideální výsledek je odhadnut extrapolací zašuměných výsledků střední hodnoty na limit nulového šumu.

Obě fáze — zesilování šumu i extrapolace — lze implementovat mnoha různými způsoby. Qiskit Runtime implementuje zesilování šumu pomocí „digitálního skládání Gate", což znamená, že dvouqubitové Gate jsou nahrazeny ekvivalentními sekvencemi Gate a jejich inverzí. Například nahrazení unitárního operátoru $U$ výrazem $U U^\dagger U$ by přineslo faktor zesílení šumu 3. Pro extrapolaci si můžeš vybrat z několika funkčních forem, včetně lineárního nebo exponenciálního fitu.
Obrázek níže zobrazuje digitální skládání Gate vlevo a postup extrapolace vpravo.

![Zobrazení ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNE lze zapnout nastavením `zne_mitigation` na `True` v [možnostech odolnosti Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pro Estimator.
Možnosti Qiskit Runtime pro ZNE jsou popsány [zde](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Následující možnosti jsou pozoruhodné:

- `noise_factors`: Faktory šumu použité pro zesilování šumu.
- `extrapolator`: Funkční forma použitá pro extrapolaci.

Následující buňka kódu ukazuje, jak zapnout ZNE a nastavit tyto možnosti pro estimator. Žádná z těchto možností nemusí být nastavena explicitně.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Probabilistické zesilování chyb (PEA)
Jednou z hlavních výzev ZNE je přesné zesilování šumu ovlivňujícího cílový Circuit. Skládání Gate poskytuje snadný způsob, jak toto zesilování provést, ale může být nepřesné a vést k nesprávným výsledkům. Viz článek [„Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), konkrétně strana 4 doplňkových informací pro podrobnosti. Probabilistické zesilování chyb poskytuje přesnější přístup k zesilování chyb prostřednictvím učení šumu.

PEA je sofistikovanější technika, která provádí předběžné experimenty k rekonstrukci šumu a poté tyto informace využívá k provedení přesného zesilování. Začíná učením twirled modelu šumu každé vrstvy provazujících Gate v Circuit před jejich spuštěním (viz [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) pro příslušné možnosti učení). Po fázi učení jsou Circuit spouštěny při každém faktoru šumu, kde každá provazující vrstva Circuit je zesilována probabilistickým vkládáním jednoqubitového šumu proporcionálního k odpovídajícímu naučenému modelu šumu. Viz článek [„Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) pro více podrobností.

PEA se skládá ze tří fází:
1. _Učení_: Je se naučen twirled model šumu každé vrstvy provazujících Gate v Circuit.
1. _Zesilování šumu_: Původní kvantový Circuit se spouští vícekrát při různých faktorech šumu.
2. _Extrapolace_: Ideální výsledek je odhadnut extrapolací zašuměných výsledků střední hodnoty na limit nulového šumu.

Pro experimenty v utility měřítku je PEA často nejlepší volbou.

Protože PEA je technika zesilování šumu ZNE, musíš také zapnout ZNE nastavením `resilience.zne_mitigation = True`. Další možnosti [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) lze navíc použít k nastavení extrapolátorů, úrovní zesilování a podobně. PEA vyžaduje model šumu, který se automaticky generuje při použití primitivů.

Následující úryvek kódu poskytuje příklad, kde PEA slouží ke zmírnění výsledku úlohy Estimatoru:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Probabilistické rušení chyb (PEC)
Probabilistické rušení chyb (PEC) je technika pro zmírnění chyb při odhadování střední hodnoty observabil. Na rozdíl od ZNE vrací nestranný odhad střední hodnoty. Obecně však přináší větší režii.

V PEC je efekt ideálního cílového Circuit vyjádřen jako lineární kombinace zašuměných Circuit, které jsou v praxi skutečně realizovatelné:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

Výstup ideálního Circuit lze pak reprodukovat spouštěním různých instancí zašuměných Circuit čerpaných z náhodného souboru definovaného lineární kombinací. Pokud koeficienty $\eta_i$ tvoří rozdělení pravděpodobnosti, lze je přímo použít jako pravděpodobnosti souboru. V praxi jsou některé koeficienty záporné, takže tvoří quasi-pravděpodobnostní rozdělení. Stále je lze použít k definování náhodného souboru, ale existuje vzorkovací režie související s negativitou quasi-pravděpodobnostního rozdělení, která je charakterizována veličinou

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Vzorkovací režie je multiplikativní faktor počtu vzorků potřebných k odhadu střední hodnoty s danou přesností ve srovnání s počtem vzorků, které by byly potřebné z ideálního Circuit. Škáluje se kvadraticky s $\gamma$, které se zase škáluje exponenciálně s hloubkou Circuit.

PEC lze zapnout nastavením `pec_mitigation` na `True` v [možnostech odolnosti Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pro Estimator.
Možnosti Qiskit Runtime pro PEC jsou popsány [zde](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Limit vzorkovací režie lze nastavit pomocí možnosti `max_overhead`. Vezmi na vědomí, že omezení vzorkovací režie může způsobit, že přesnost výsledku překročí požadovanou přesnost. Výchozí hodnota `max_overhead` je 100.

Následující buňka kódu ukazuje, jak zapnout PEC a nastavit možnost `max_overhead` pro estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Další kroky
- Prohlédni si [tutoriál](/tutorials/combine-error-mitigation-techniques) o kombinování možností zmírnění chyb s primitivem Estimator.
- [Nakonfiguruj zmírnění chyb.](configure-error-mitigation)
- [Nakonfiguruj potlačení chyb.](configure-error-suppression)
- Prozkoumej další [možnosti](runtime-options-overview) pro primitiva Qiskit Runtime.
- Rozhodni se, v jakém [režimu spouštění](execution-modes) spustíš svou úlohu.